# Assignment 2 — IMDB Sentiment Analysis



In [ ]:
# Install dependencies
!pip -q install scikit-learn tensorflow pandas numpy matplotlib seaborn

In [ ]:
import os
print("Files in current directory:")
print(os.listdir())

Files in current directory:
['.config', 'IMDB Dataset.csv', '.ipynb_checkpoints', 'sample_data']


In [ ]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv", encoding="utf-8", engine="python", on_bad_lines="skip")
df = df[['review', 'sentiment']].dropna().copy()

df['review'] = df['review'].astype(str)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("Shape:", df.shape)
df.head()

Shape: (50000, 3)


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


## Step 2 — Basic preprocessing

We keep preprocessing simple:
- lowercase
- remove HTML tags
- remove punctuation
- remove extra spaces

This is enough for a strong assignment without making the notebook too complicated.

In [ ]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)                      # remove HTML
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)                        # remove numbers
    text = re.sub(r'\s+', ' ', text).strip()                # remove extra spaces
    return text

df['clean_review'] = df['review'].apply(clean_text)
df[['review', 'clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love in the time of money is a ...


## Step 3 — Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 40000
Test size: 10000


## Step 4 — Model 1: TF-IDF + Logistic Regression

This is one of the strongest simple baselines for sentiment classification.
We also tune the main parameters to improve performance.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

logreg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

logreg_params = {
    'tfidf__max_features': [10000, 20000],
    'tfidf__ngram_range': [(1,1), (1,2)],
    'clf__C': [0.5, 1.0, 2.0]
}

logreg_grid = GridSearchCV(
    logreg_pipeline,
    logreg_params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

logreg_grid.fit(X_train, y_train)

print("Best Logistic Regression Params:")
print(logreg_grid.best_params_)
print("Best CV Score:", logreg_grid.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Logistic Regression Params:
{'clf__C': 2.0, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}
Best CV Score: 0.8909749854890814


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

best_logreg = logreg_grid.best_estimator_
logreg_pred = best_logreg.predict(X_test)

logreg_acc = accuracy_score(y_test, logreg_pred)
print("Logistic Regression Test Accuracy:", logreg_acc)
print(classification_report(y_test, logreg_pred))

Logistic Regression Test Accuracy: 0.9023
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      5000
           1       0.90      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



## Step 5 — Model 2: TF-IDF + Linear SVM

Linear SVM is often one of the best models for text classification with TF-IDF.

In [ ]:
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LinearSVC())
])

svm_params = {
    'tfidf__max_features': [10000, 20000],
    'tfidf__ngram_range': [(1,1), (1,2)],
    'clf__C': [0.5, 1.0, 2.0]
}

svm_grid = GridSearchCV(
    svm_pipeline,
    svm_params,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train, y_train)

print("Best SVM Params:")
print(svm_grid.best_params_)
print("Best CV Score:", svm_grid.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best SVM Params:
{'clf__C': 0.5, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}
Best CV Score: 0.8901249736133159


In [ ]:
best_svm = svm_grid.best_estimator_
svm_pred = best_svm.predict(X_test)

svm_acc = accuracy_score(y_test, svm_pred)
print("Linear SVM Test Accuracy:", svm_acc)
print(classification_report(y_test, svm_pred))

Linear SVM Test Accuracy: 0.9009
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      5000
           1       0.89      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



## Step 6 — Model 3: Bidirectional LSTM

This model satisfies the assignment suggestion to train an RNN variant.
We keep the architecture simple but effective:
- Tokenizer
- Padding
- Embedding
- Bidirectional LSTM
- Dense layers

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

max_words = 20000
max_len = 200

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print("Padded train shape:", X_train_pad.shape)
print("Padded test shape:", X_test_pad.shape)

Padded train shape: (40000, 200)
Padded test shape: (10000, 200)


In [ ]:
bilstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = bilstm_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=6,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 187s 728ms/step - accuracy: 0.7713 - loss: 0.4634 - val_accuracy: 0.8727 - val_loss: 0.3252
Epoch 2/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 178s 714ms/step - accuracy: 0.9030 - loss: 0.2552 - val_accuracy: 0.8761 - val_loss: 0.3154
Epoch 3/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 184s 738ms/step - accuracy: 0.9257 - loss: 0.2075 - val_accuracy: 0.8512 - val_loss: 0.4073
Epoch 4/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 177s 710ms/step - accuracy: 0.9355 - loss: 0.1797 - val_accuracy: 0.8224 - val_loss: 0.7087


In [ ]:
bilstm_loss, bilstm_acc = bilstm_model.evaluate(X_test_pad, y_test, verbose=0)
print("BiLSTM Test Accuracy:", bilstm_acc)

BiLSTM Test Accuracy: 0.8654000163078308


In [ ]:
bilstm_pred_prob = bilstm_model.predict(X_test_pad, verbose=0)
bilstm_pred = (bilstm_pred_prob > 0.5).astype(int)

print(classification_report(y_test, bilstm_pred))

              precision    recall  f1-score   support

           0       0.87      0.86      0.86      5000
           1       0.86      0.87      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



## Step 7 — Compare all models

In [ ]:
results = pd.DataFrame({
    'Model': ['TF-IDF + Logistic Regression', 'TF-IDF + Linear SVM', 'Bidirectional LSTM'],
    'Accuracy': [logreg_acc, svm_acc, bilstm_acc]
}).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

results

,Model,Accuracy
0,TF-IDF + Logistic Regression,0.9023
1,TF-IDF + Linear SVM,0.9009
2,Bidirectional LSTM,0.8654


## Step 8 — Save results for GitHub submission

In [ ]:
import json
import os

os.makedirs("assignment2_results", exist_ok=True)

results.to_csv("assignment2_results/model_comparison.csv", index=False)

with open("assignment2_results/logreg_results.txt", "w", encoding="utf-8") as f:
    f.write("Best Parameters:\n")
    f.write(str(logreg_grid.best_params_) + "\n\n")
    f.write("Test Accuracy:\n")
    f.write(str(logreg_acc) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_test, logreg_pred))

with open("assignment2_results/svm_results.txt", "w", encoding="utf-8") as f:
    f.write("Best Parameters:\n")
    f.write(str(svm_grid.best_params_) + "\n\n")
    f.write("Test Accuracy:\n")
    f.write(str(svm_acc) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_test, svm_pred))

with open("assignment2_results/bilstm_results.txt", "w", encoding="utf-8") as f:
    f.write("Architecture:\n")
    f.write("Embedding -> Bidirectional LSTM(64) -> Dropout -> Dense(64) -> Dropout -> Sigmoid\n\n")
    f.write("Test Accuracy:\n")
    f.write(str(bilstm_acc) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_test, bilstm_pred))

readme_text = '''
# Assignment 2 — IMDB Sentiment Analysis

## Models
1. TF-IDF + Logistic Regression
2. TF-IDF + Linear SVM
3. Bidirectional LSTM

## Techniques Applied
- Text cleaning
- TF-IDF vectorization
- Hyperparameter tuning using GridSearchCV
- Deep learning with a recurrent neural network (BiLSTM)
- Early stopping to reduce overfitting

## Conclusion
This assignment improves on Assignment 1 by replacing the small hand-engineered feature set with stronger text representations and more powerful models.
'''.strip()

with open("assignment2_results/README_submission.txt", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("Saved files:")
print(os.listdir("assignment2_results"))

Saved files:
['bilstm_results.txt', 'model_comparison.csv', 'svm_results.txt', 'README_submission.txt', 'logreg_results.txt']


## Step 9 — Save best models



In [ ]:
import joblib

joblib.dump(best_logreg, "assignment2_results/best_logreg.pkl")
joblib.dump(best_svm, "assignment2_results/best_svm.pkl")
bilstm_model.save("assignment2_results/best_bilstm.keras")

print("Models saved successfully.")

Models saved successfully.
